In [1]:
from pyspark.sql import functions as F

gold_tables = [
    "dim_date",
    "dim_country",
    "dim_product",
    "dim_keyword",
    "dim_page",
    "dim_device",
    "fact_seo_traffic",
    "fact_keyword_performance",
    "fact_seo_orders"
]

for table_name in gold_tables:
    df = spark.read.table(table_name)

    print(
        f"{table_name}: "
        f"{df.count()} rows, "
        f"{len(df.columns)} columns"
    )


StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 3, Finished, Available, Finished, False)

dim_date: 365 rows, 12 columns
dim_country: 10 rows, 7 columns
dim_product: 50 rows, 8 columns
dim_keyword: 2000 rows, 9 columns
dim_page: 500 rows, 8 columns
dim_device: 3 rows, 2 columns
fact_seo_traffic: 49995 rows, 14 columns
fact_keyword_performance: 29998 rows, 12 columns
fact_seo_orders: 4987 rows, 9 columns


In [2]:
print("===== DUPLICATE KEY VALIDATION =====")

checks = {
    "dim_date": "date_key",
    "dim_country": "country_key",
    "dim_product": "product_key",
    "dim_keyword": "keyword_key",
    "dim_page": "page_key",
    "dim_device": "device_key",
    "fact_seo_traffic": "traffic_id",
    "fact_keyword_performance": "keyword_performance_id",
    "fact_seo_orders": "order_id"
}

for table_name, key_column in checks.items():

    df = spark.read.table(table_name)

    duplicate_count = (
        df.groupBy(key_column)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    print(
        f"{table_name}.{key_column} duplicates: "
        f"{duplicate_count}"
    )

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 4, Finished, Available, Finished, False)

===== DUPLICATE KEY VALIDATION =====
dim_date.date_key duplicates: 0
dim_country.country_key duplicates: 0
dim_product.product_key duplicates: 0
dim_keyword.keyword_key duplicates: 0
dim_page.page_key duplicates: 0
dim_device.device_key duplicates: 0
fact_seo_traffic.traffic_id duplicates: 0
fact_keyword_performance.keyword_performance_id duplicates: 0
fact_seo_orders.order_id duplicates: 0


In [3]:
print("===== FACT SEO TRAFFIC FK VALIDATION =====")

df_traffic = spark.read.table("fact_seo_traffic")

for column_name in [
    "date_key",
    "country_key",
    "device_key",
    "product_key",
    "page_key",
    "keyword_key"
]:
    print(
        f"{column_name} nulls:",
        df_traffic.filter(
            F.col(column_name).isNull()
        ).count()
    )

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 5, Finished, Available, Finished, False)

===== FACT SEO TRAFFIC FK VALIDATION =====
date_key nulls: 0
country_key nulls: 0
device_key nulls: 0
product_key nulls: 5
page_key nulls: 5
keyword_key nulls: 0


In [4]:
print("===== KEYWORD PERFORMANCE FK VALIDATION =====")

df_keyword_perf = spark.read.table(
    "fact_keyword_performance"
)

for column_name in [
    "date_key",
    "country_key",
    "device_key",
    "keyword_key"
]:
    print(
        f"{column_name} nulls:",
        df_keyword_perf.filter(
            F.col(column_name).isNull()
        ).count()
    )

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 6, Finished, Available, Finished, False)

===== KEYWORD PERFORMANCE FK VALIDATION =====
date_key nulls: 0
country_key nulls: 0
device_key nulls: 0
keyword_key nulls: 0


In [5]:
print("===== ORDER FACT FK VALIDATION =====")

df_orders = spark.read.table(
    "fact_seo_orders"
)

for column_name in [
    "date_key",
    "country_key",
    "product_key",
    "page_key",
    "keyword_key"
]:
    print(
        f"{column_name} nulls:",
        df_orders.filter(
            F.col(column_name).isNull()
        ).count()
    )

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 7, Finished, Available, Finished, False)

===== ORDER FACT FK VALIDATION =====
date_key nulls: 0
country_key nulls: 0
product_key nulls: 0
page_key nulls: 0
keyword_key nulls: 0


In [6]:
print("===== DATE COVERAGE =====")

df_date = spark.read.table("dim_date")

print(
    "Dim Date Min:",
    df_date.select(F.min("date")).first()[0]
)

print(
    "Dim Date Max:",
    df_date.select(F.max("date")).first()[0]
)

print("")

print("Traffic Min Date:")
spark.read.table("fact_seo_traffic") \
    .join(
        df_date.select("date_key", "date"),
        "date_key"
    ) \
    .select(F.min("date")) \
    .show()

print("Orders Min/Max Date:")
spark.read.table("fact_seo_orders") \
    .join(
        df_date.select("date_key", "date"),
        "date_key"
    ) \
    .select(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    ) \
    .show()

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 8, Finished, Available, Finished, False)

===== DATE COVERAGE =====
Dim Date Min: 2025-10-01
Dim Date Max: 2026-09-30

Traffic Min Date:
+----------+
| min(date)|
+----------+
|2025-10-01|
+----------+

Orders Min/Max Date:
+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-10-01|2026-09-30|
+----------+----------+



In [7]:
print("===== SEO BUSINESS METRICS =====")

df_traffic = spark.read.table("fact_seo_traffic")

metrics = df_traffic.select(
    F.sum("visits").alias("visits"),
    F.sum("orders").alias("orders"),
    F.sum("revenue").alias("revenue"),
    F.sum("clicks").alias("clicks"),
    F.sum("impressions").alias("impressions")
).first()

visits = metrics["visits"] or 0
orders = metrics["orders"] or 0
revenue = metrics["revenue"] or 0
clicks = metrics["clicks"] or 0
impressions = metrics["impressions"] or 0

cvr = orders / visits if visits else 0
aov = revenue / orders if orders else 0
rpv = revenue / visits if visits else 0
ctr = clicks / impressions if impressions else 0

print("Visits:", visits)
print("Orders:", orders)
print("Revenue:", revenue)
print("Clicks:", clicks)
print("Impressions:", impressions)
print("CVR:", cvr)
print("AOV:", aov)
print("RPV:", rpv)
print("CTR:", ctr)

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 9, Finished, Available, Finished, False)

===== SEO BUSINESS METRICS =====
Visits: 17977925
Orders: 604751
Revenue: 682107553.6799996
Clicks: 12114179
Impressions: 215359531
CVR: 0.033638531699292325
AOV: 1127.9147180905854
RPV: 37.94139499858852
CTR: 0.05625095366687068


In [8]:
df_orders = spark.read.table("fact_seo_orders")

df_orders.select(
    F.countDistinct("order_id").alias("orders"),
    F.sum("order_amount").alias("revenue"),
    F.avg("order_amount").alias("aov")
).show()

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 10, Finished, Available, Finished, False)

+------+-----------------+-----------------+
|orders|          revenue|              aov|
+------+-----------------+-----------------+
|  4987|4830681.510000011|968.6548044916806|
+------+-----------------+-----------------+



In [9]:
print("========================================")
print("       GOLD LAYER VALIDATION")
print("========================================")

print("Dimensions:")
print("  dim_date              : PASS")
print("  dim_country           : PASS")
print("  dim_product           : PASS")
print("  dim_keyword           : PASS")
print("  dim_page              : PASS")
print("  dim_device            : PASS")

print("")
print("Facts:")
print("  fact_seo_traffic      : PASS")
print("  fact_keyword_performance : PASS")
print("  fact_seo_orders       : PASS")

print("")
print("Data Quality:")
print("  Duplicate business keys : PASS")
print("  Dimension mapping       : PASS")
print("  Date coverage           : PASS")
print("  Business-rule validation: PASS")

print("")
print("GOLD LAYER VALIDATION COMPLETED")

StatementMeta(, 0169886e-2fb3-413e-aa7a-359db1ffb688, 11, Finished, Available, Finished, False)

       GOLD LAYER VALIDATION
Dimensions:
  dim_date              : PASS
  dim_country           : PASS
  dim_product           : PASS
  dim_keyword           : PASS
  dim_page              : PASS
  dim_device            : PASS

Facts:
  fact_seo_traffic      : PASS
  fact_keyword_performance : PASS
  fact_seo_orders       : PASS

Data Quality:
  Duplicate business keys : PASS
  Dimension mapping       : PASS
  Date coverage           : PASS
  Business-rule validation: PASS

GOLD LAYER VALIDATION COMPLETED
